# WaveLab RL trên Colab
Nếu cell kiểm tra Python không in thông báo, hãy kiểm tra nút Connect/Kết nối của Colab và runtime GPU. Chạy từng cell theo thứ tự; không Run all vì cell STOP ở cuối sẽ yêu cầu dừng train.


In [ ]:
print('Python đang hoạt động. Tiếp theo sẽ yêu cầu kết nối Google Drive.', flush=True)
from google.colab import drive
drive.mount('/content/drive')
print('Đã kết nối Google Drive.', flush=True)

In [ ]:
import subprocess,sys,os,json,hashlib,shutil,time
from pathlib import Path
REPO=Path('/content/wavelab-rl-scm')
def run_visible(command):
    print('Đang chạy:', ' '.join(map(str,command)), flush=True)
    child=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in child.stdout: print(line,end='',flush=True)
        status=child.wait()
        if status: raise RuntimeError(f'Lệnh thất bại, exit code {status}')
    except BaseException:
        child.terminate()
        raise
if not REPO.exists(): run_visible(['git','clone','--depth','1','https://github.com/hongphuc-git/wavelab-rl-scm.git',str(REPO)])
else: print('Repo đã có:',REPO,flush=True)
print('Hoàn tất bước tải code.', flush=True)

In [ ]:
print('Kiểm tra GPU và PyTorch có sẵn...',flush=True)
import torch,torchvision
print('Torch:',torch.__version__,'Torchvision:',torchvision.__version__,flush=True)
assert torch.cuda.is_available(), 'Chọn Runtime → Change runtime type → GPU rồi kết nối lại'
print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('Cài thư viện RL, log sẽ hiển thị bên dưới...',flush=True)
run_visible([sys.executable,'-m','pip','install','--disable-pip-version-check','--timeout','60','--retries','2','-r',str(REPO/'requirements-colab.txt')])
print('Cài đặt xong. Có thể chạy cell Cấu hình.',flush=True)

## Cấu hình
Epoch 100 mặc định. Đổi thành 50 để đối chứng. Giữ torch/torchvision có sẵn nếu import được.

In [ ]:
EPOCH=100
TOTAL_STEPS=4000
DRIVE=Path('/content/drive/MyDrive')
SOURCE=DRIVE/'conv/Bone_Phantom_Training_Data_b8.h5'
WEIGHTS=DRIVE/f'conv/18sep/pipe_efficientnet_v2_l/epoch_{EPOCH}_checkpoint/pytorch_model.bin'
RUN=DRIVE/f'WaveLab_RL/efficientnet_v2_l_epoch{EPOCH}/overfit16_seed2026'
RUN.mkdir(parents=True,exist_ok=True)
assert SOURCE.exists() and WEIGHTS.exists(), 'Kiểm tra đường dẫn Drive'
LOCAL_WEIGHTS=Path(f'/content/epoch{EPOCH}.bin')
if not LOCAL_WEIGHTS.exists(): shutil.copyfile(WEIGHTS,LOCAL_WEIGHTS)
os.environ['WAVEDL_ENCODER_CHECKPOINT']=str(LOCAL_WEIGHTS)
os.environ['RLSCM_BACKEND']='python'
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'): os.environ[name]='1'
STOP=RUN/'STOP'
os.environ['RLSCM_STOP_FILE']=str(STOP)
print('Kết quả:',RUN)

In [ ]:
import numpy as np,h5py
DATA=RUN/'train16.h5'
if not DATA.exists():
    with h5py.File(SOURCE,'r') as f:
        counts=np.asarray(f['nnz_per_matrix']).ravel()
        ids=np.sort(np.random.default_rng(20261113).choice(np.flatnonzero(counts>0),16,replace=False))
        labels=np.asarray(f['samples'])[:,ids].T
        H=int(np.asarray(f.attrs['grid_H']).item()); W=int(np.asarray(f.attrs['grid_W']).item())
        images=np.zeros((16,256,256),np.uint8)
        for j,i in enumerate(ids):
            n=int(counts[i]);r=f['row_indices'][i,:n].astype(np.int64);c=f['col_indices'][i,:n].astype(np.int64)
            images[j,np.minimum(r*256//H,255),np.minimum(c*256//W,255)]=1
        with h5py.File(DATA,'w') as g:
            g.attrs['split']='train';g.create_dataset('images',data=images);g.create_dataset('parameters',data=labels);g.create_dataset('complete',data=np.ones(16,bool));g.create_dataset('source_indices',data=ids)
    manifest=dict(source=str(SOURCE),source_indices=ids.tolist(),geometry='free pipe b=8 mm',preprocessing='Original WaveDL floor mapping; binary 0/1 256x256',checkpoint=str(WEIGHTS),checkpoint_sha256=hashlib.sha256(LOCAL_WEIGHTS.read_bytes()).hexdigest(),diagnostic='16 train samples, not test generalization')
    (RUN/'manifest.json').write_text(json.dumps(manifest,indent=2))
print('Dataset:',DATA)

## Bắt đầu hoặc resume
Train chạy nền. Không chạy cell hai lần khi tiến trình còn sống. Mỗi 1k bước lưu model và replay vào Drive. Colab có thể mất kết nối; dùng cell này để resume.

In [ ]:
if 'process' in globals() and process.poll() is None: raise RuntimeError('Đang train')
checkpoints=sorted(RUN.glob('checkpoint_*_steps.zip'),key=lambda p:int(p.stem.split('_')[1]))
latest=checkpoints[-1] if checkpoints else None
done=int(latest.stem.split('_')[1]) if latest else 0
assert done<TOTAL_STEPS, 'Đã đạt ngân sách'
STOP.unlink(missing_ok=True)
initial_lr=1e-4;final_lr=9.76e-5
resume_lr=initial_lr+(final_lr-initial_lr)*done/TOTAL_STEPS
cmd=[sys.executable,'-u',str(REPO/'scripts/train_bone_sac_parallel.py'),'--dataset',str(DATA),'--steps',str(TOTAL_STEPS-done),'--n-envs','2','--encoder',f'wavedl_epoch{EPOCH}','--share-encoder','--param-weight','2','--horizon','8','--action-scale','0.15','--buffer-size','2000','--batch-size','8','--learning-starts','128','--learning-rate',str(resume_lr),'--lr-final',str(final_lr),'--save-every','1000','--save-replay','--seed','2026','--device','cuda','--torch-threads','2','--out',str(RUN)]
if latest: cmd+=['--resume',str(latest)]
else: cmd+=['--entropy-coef','0.01','--save-initial']
log=(RUN/'training.log').open('a')
process=subprocess.Popen(cmd,cwd=REPO,stdout=log,stderr=subprocess.STDOUT,env=os.environ.copy())
print('PID',process.pid,'resume từ',done,'mục tiêu',TOTAL_STEPS)

In [ ]:
print((RUN/'training.log').read_text()[-5000:])
if (RUN/'progress.json').exists():print((RUN/'progress.json').read_text())
print('Exit code:',process.poll() if 'process' in globals() else 'Phiên Colab mới')

## Đánh giá checkpoint trên 16 mẫu train
Cùng seed khởi tạo; nhãn chỉ dùng báo cáo.

In [ ]:
# Chạy sau khi train kết thúc. Đánh giá trên chính 16 mẫu, không phải test độc lập.
assert 'process' not in globals() or process.poll() is not None, 'Chờ train kết thúc'
sys.path.insert(0,str(REPO))
from stable_baselines3 import SAC
from rlscm.backend import SCM
from rlscm.environment import PipeEnv
from rlscm.paths import LOW,HIGH
EVAL=RUN/'evaluation';EVAL.mkdir(exist_ok=True)
with h5py.File(DATA) as f: images=f['images'][:];truth=f['parameters'][:]
backend=SCM(EVAL,20,256,curve_points=256)
reports=[]
try:
    for cp in sorted(RUN.glob('checkpoint_*_steps.zip'),key=lambda p:int(p.stem.split('_')[1])):
        import gc
        if 'model' in globals():del model
        gc.collect();torch.cuda.empty_cache()
        model=SAC.load(cp,device='cuda',buffer_size=1)
        cases=[]
        for i,img in enumerate(images):
            env=PipeEnv(backend,[{'image':img.astype(np.int8)}],horizon=8,image_score=True,action_scale=.15,score_sigmas=(4,12))
            obs,_=env.reset(seed=20261113+i)
            initial=(LOW+env.x*(HIGH-LOW)).copy()
            for _ in range(8):obs,*_=env.step(model.predict(obs,deterministic=True)[0])
            cases.append(dict(case=i,initial=initial.tolist(),prediction=env.best_m.tolist(),truth=truth[i].tolist(),J=env.best_J))
        error=abs(np.array([c['prediction'] for c in cases])-truth)
        row=dict(checkpoint=cp.name,checkpoint_sha256=hashlib.sha256(cp.read_bytes()).hexdigest(),mean_normalized_error=float((error/(HIGH-LOW)).mean()),mare_percent=(100*error/abs(truth)).mean(0).tolist(),cases=cases)
        reports.append(row)
        (EVAL/'results.json').write_text(json.dumps(reports,indent=2))
        print(cp.name,row['mean_normalized_error'],row['mare_percent'])
finally:backend.close()


## Dừng an toàn
Chỉ chạy cell dưới khi muốn dừng. Chờ log STOPPED và tiến trình thoát trước khi tắt runtime.

In [ ]:
STOP.touch()
print('Đã yêu cầu lưu checkpoint + replay và dừng; kiểm tra log trước khi tắt runtime.')